# Prisoner's Dilemma — experiment template

Copy this notebook, rename it, and change the CONFIG cell to run a
different experiment. Everything below the CONFIG cell is generic
plumbing you can leave untouched.

**Reproducibility:** the `Game` is constructed with an explicit
`random.Random(seed)` instance. Any player that needs its own stream
should be given a separate `random.Random(...)` in its constructor.

**Editing library code:** the `pd` package is installed with `pip
install -e .[dev]`, so edits in `src/pd/` are picked up on the next
cell run — no reinstall needed. The `%load_ext autoreload` cell below
makes changes visible without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt

from pd import (
    AlwaysCooperate,
    ClassicAxelrodGenerator,
    Game,
    TitForTat,
)

## CONFIG

Change these two things (players, rounds) and rerun the notebook.

In [ ]:
SEED = 42
TOTAL_ROUNDS = 200

# Extend this list to bring in more strategies. Any Player subclass
# from pd.players.* works here. Strategies that need randomness take
# an rng argument, e.g. RandomDefect(p=0.3, rng=random.Random("rd")).
players = [
    TitForTat(),
    TitForTat(),
    AlwaysCooperate(),
    AlwaysCooperate(),
    AlwaysCooperate(),
]

deal_generator = ClassicAxelrodGenerator()
rng = random.Random(SEED)

## Run the game

In [ ]:
game = Game(deal_generator=deal_generator, players=players,
            total_rounds=TOTAL_ROUNDS, rng=rng)
game.play()

print(f"Total deals: {len(game.history)}")
print("Leaderboard:")
for player, score in game.leaderboard():
    label = f"{player.name()}#{player.player_id}"
    print(f"  {label:>22}  {score:>8.1f}")

## Cumulative score per player, per round

Multiline chart: one curve per player, x = round index, y = score
accumulated up to and including that round.

In [ ]:
def cumulative_scores_by_round(game):
    """Return dict[player_id] -> np.ndarray of shape (TOTAL_ROUNDS,)
    with cumulative score after each round."""
    per_round = {p.player_id: np.zeros(game.total_rounds) for p in game.players}
    for deal in game.history:
        per_round[deal.player_1.player_id][deal.round_index] += deal.score_1
        per_round[deal.player_2.player_id][deal.round_index] += deal.score_2
    return {pid: np.cumsum(arr) for pid, arr in per_round.items()}

cum = cumulative_scores_by_round(game)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(game.total_rounds)
for player in game.players:
    label = f"{player.name()}#{player.player_id}"
    ax.plot(x, cum[player.player_id], label=label, linewidth=1.5)
ax.set_xlabel("round")
ax.set_ylabel("cumulative score")
ax.set_title("Cumulative score per player")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Two-axis chart: total score vs cooperation rate

Left y-axis: final score. Right y-axis: fraction of that player's
deals in which they played COOPERATE. Ranks each player on the x-axis.

In [ ]:
from pd import Action

def cooperation_rate(player):
    if not player.history:
        return 0.0
    coops = 0
    for d in player.history:
        act = d.action_1 if d.player_1 is player else d.action_2
        if act == Action.COOPERATE:
            coops += 1
    return coops / len(player.history)

ordered = [p for p, _ in game.leaderboard()]
labels = [f"{p.name()}#{p.player_id}" for p in ordered]
scores = [p.total_score() for p in ordered]
coop_rates = [cooperation_rate(p) for p in ordered]

fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(ordered))
bars = ax1.bar(x, scores, alpha=0.6, label="total score")
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=30, ha="right")
ax1.set_ylabel("total score")
ax1.grid(True, axis="y", alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(x, coop_rates, color="crimson", marker="o",
         linewidth=2, label="cooperation rate")
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("cooperation rate")

ax1.set_title("Score and cooperation rate per player")
fig.tight_layout()
plt.show()

## Heatmap: mean score per (row, column) matchup

Cell (i, j) = mean score of player i across all deals in which player i
faced player j. Diagonal is empty (players don't play themselves).

In [ ]:
def pairwise_mean_score_matrix(game):
    n = len(game.players)
    sums = np.zeros((n, n))
    counts = np.zeros((n, n), dtype=int)
    for deal in game.history:
        i = deal.player_1.player_id
        j = deal.player_2.player_id
        sums[i, j] += deal.score_1
        counts[i, j] += 1
        sums[j, i] += deal.score_2
        counts[j, i] += 1
    with np.errstate(divide="ignore", invalid="ignore"):
        mean = np.where(counts > 0, sums / counts, np.nan)
    return mean

mat = pairwise_mean_score_matrix(game)
labels = [f"{p.name()}#{p.player_id}" for p in game.players]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat, cmap="viridis", aspect="auto")
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("opponent")
ax.set_ylabel("player (row)")
ax.set_title("Mean per-deal score of row-player when facing column-player")

for i in range(len(labels)):
    for j in range(len(labels)):
        v = mat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < np.nanmean(mat) else "black",
                    fontsize=8)
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

## Where to go next

- Change `players` in the CONFIG cell and rerun everything.
- Add a new strategy in `src/pd/players/your_strategy.py`, re-export it
  in `pd/players/__init__.py` and `pd/__init__.py`, then import it here.
  Autoreload picks it up without a kernel restart.
- For throwaway scratch work that shouldn't hit git, use the `junk/`
  directory — it's gitignored.